# 5.1 — Reading the Spark UI

**Chapter 5, section 5.2** (the Spark web UI) and **section 5.8.1** (partitioning on write),
and the starting point for **Exercise 1**.

**The question this notebook answers:** the chapter's worked diagnosis is a job of three lines
that takes about a minute to process 273 MB, and the chapter walks the Spark UI from the Jobs
tab down to the executor to find out why. This notebook runs that diagnosis for real — it
builds a badly laid-out input, runs the job, reads every tab the chapter names, applies the
remedy, and prints the two sets of summary metrics beside each other.

**How a headless notebook reads a web UI.** It does not. The Spark UI is a rendering of a JSON
API that the driver serves in its own process, and every page the chapter names has an endpoint
behind it. This notebook queries those endpoints, so the numbers below are the same numbers the
browser would draw, and they are produced by running the job rather than quoted from a
screenshot.

Runs on a laptop in well under a minute. Every quantity it prints is machine-dependent; the
*ratios* between the two runs are the part that transfers.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, json, glob, time, shutil, tempfile, logging, urllib.request
from urllib.parse import urlparse
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-5.1")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

print("Spark", spark.version)
print("slots on this machine (executors x cores):", sc.defaultParallelism)

Spark 4.2.0
slots on this machine (executors x cores): 18


## The instrument: the Spark UI is a JSON API

`sc.uiWebUrl` is the address the chapter tells you to open in a browser. Appending `/api/v1`
to it gives the same information as JSON, from the same in-process server — no data leaves this
machine and nothing is downloaded.

One correction is applied below. `uiWebUrl` reports the driver's address on the local network,
which stops resolving the moment the machine changes network. The port is the part that
matters, so the notebook keeps the port and asks `localhost` for it.

| Chapter's page | Endpoint used below |
|---|---|
| Jobs tab (§5.2.3) | `/jobs` |
| Stages tab (§5.2.4) | `/stages` |
| **summary metrics** (§5.2.4) | `/stages/<id>/<attempt>/taskSummary?quantiles=…` |
| SQL tab (§5.2.5) | `/sql` |
| Executors tab (§5.2.6) | `/executors` |
| Environment tab (§5.2.6) | `/environment` |
| Storage tab (§5.9.3) | `/storage/rdd` |

In [2]:
_port = urlparse(sc.uiWebUrl).port
UI = f"http://localhost:{_port}/api/v1"
APP = json.load(urllib.request.urlopen(f"{UI}/applications"))[0]["id"]

def ui(path):
    """One endpoint of this driver's own Spark UI, as JSON."""
    with urllib.request.urlopen(f"{UI}/applications/{APP}{path}", timeout=60) as r:
        return json.load(r)

print("browser would open :", sc.uiWebUrl)
print("this notebook asks :", UI)
print("application id     :", APP)
print("endpoints answering:",
      {p: len(ui("/" + p)) for p in ("jobs", "stages", "executors")})

browser would open : http://172.20.10.2:4040
this notebook asks : http://localhost:4040/api/v1
application id     : local-1789220663564
endpoints answering: {'jobs': 0, 'stages': 0, 'executors': 1}


## 1. The pathological input

The chapter's diagnosis ends at the files: *"the input consists of a very large number of very
small files."* That input is built here from the GDELT extract, written out as 1,200 fragments of
roughly sixteen kilobytes each. The distinct-value count of the column the output will be
partitioned by is measured rather than assumed, because it is one of the two numbers that decide
how many files the write produces.

Nothing about the data is unusual. Only its layout is.

In [3]:
SOURCE = os.path.join(DATA, "gdelt", "gdelt-events-small.parquet")
COLS = ["GLOBALEVENTID", "SQLDATE", "EventRootCode", "QuadClass",
        "NumMentions", "AvgTone", "Actor1Name", "ActionGeo_CountryCode"]

MANY = os.path.join(SCRATCH, "ch05-many-small-files")
ONE  = os.path.join(SCRATCH, "ch05-one-file")
for p in (MANY, ONE):
    shutil.rmtree(p, ignore_errors=True)

src = spark.read.parquet(SOURCE).select(*COLS)
n_rows = src.count()

sc.setJobDescription("build the badly laid out input")
src.repartition(1200).write.mode("overwrite").parquet(MANY)
sc.setJobDescription("build the same data as one file")
src.repartition(1).write.mode("overwrite").parquet(ONE)

def folder_stats(path):
    fs = glob.glob(os.path.join(path, "**", "*.parquet"), recursive=True)
    total = sum(os.path.getsize(f) for f in fs)
    return len(fs), total

n_many, b_many = folder_stats(MANY)
n_one,  b_one  = folder_stats(ONE)
n_dates = src.select("SQLDATE").distinct().count()

print(f"rows in the extract  : {n_rows:,}  ({len(COLS)} columns kept)")
print(f"distinct SQLDATE     : {n_dates}   <- the write below partitions by this column")
print(f"badly laid out       : {n_many:>5,} files, {b_many/1e6:6.1f} MB total,"
      f" {b_many/n_many/1e3:7.1f} KB each")
print(f"the same data, once  : {n_one:>5,} file,  {b_one/1e6:6.1f} MB total,"
      f" {b_one/n_one/1e3:7.1f} KB each")
print(f"fragmentation cost   : {b_many/b_one:.1f}x the bytes, for exactly the same rows --"
      f" Parquet compresses and encodes per file, and 16 KB is too little to work with")

rows in the extract  : 731,884  (8 columns kept)
distinct SQLDATE     : 35   <- the write below partitions by this column
badly laid out       : 1,200 files,   19.6 MB total,    16.4 KB each
the same data, once  :     1 file,     7.4 MB total,  7379.8 KB each
fragmentation cost   : 2.7x the bytes, for exactly the same rows -- Parquet compresses and encodes per file, and 16 KB is too little to work with


## 2. What the reader actually does with 1,200 files

Before running the job, the thing to establish is how many tasks the read will create, because
the chapter's first symptom row is *thousands of tasks, each of a few milliseconds*.

The chapter states the mechanism at §5.7.2: *"every file contributes at least one split, so the
partition count is bounded below by the file count."* That is a checkable claim, and the cell
below checks it.

In [4]:
part_many = spark.read.parquet(MANY).rdd.getNumPartitions()
part_one  = spark.read.parquet(ONE).rdd.getNumPartitions()

print(f"{n_many:>5,} files -> {part_many:>4} read partitions")
print(f"{n_one:>5,} file  -> {part_one:>4} read partitions")
print()
print("If the partition count were bounded below by the file count, the first line")
print(f"would read {n_many:,}.  It reads {part_many}.")
print("The second line is the same rule seen from the other side: one file larger than the")
print("budget is *split*, so the count is not bounded above by the file count either.")

1,200 files ->   38 read partitions
    1 file  ->    2 read partitions

If the partition count were bounded below by the file count, the first line
would read 1,200.  It reads 38.
The second line is the same rule seen from the other side: one file larger than the
budget is *split*, so the count is not bounded above by the file count either.


### The reader packs small files; it does not take one per file

Spark's file-based reader sorts the files by size and packs them into partitions up to a budget,
charging each file a fixed opening cost as well as its bytes. Two properties set the budget:
`spark.sql.files.maxPartitionBytes`, the 128 MB cap the chapter names, and
`spark.sql.files.openCostInBytes`, which the chapter does not mention and which is what decides
the outcome when the files are small.

The budget per partition is

$$\texttt{maxSplitBytes} \;=\; \min\Big(\texttt{maxPartitionBytes},\;
\max\big(\texttt{openCostInBytes},\; \tfrac{\text{total bytes} \;+\; \text{files}\times\texttt{openCostInBytes}}{\text{slots}}\big)\Big)$$

and each file consumes `max(its size, openCostInBytes)` of it. With files far below the opening
cost, the arithmetic collapses to *files per partition = maxSplitBytes / openCostInBytes*, and
the byte sizes drop out entirely.

In [5]:
import math
OPEN_COST = int(spark.conf.get("spark.sql.files.openCostInBytes").rstrip("b"))
MAX_PART  = int(spark.conf.get("spark.sql.files.maxPartitionBytes").rstrip("b"))
slots     = sc.defaultParallelism

bytes_per_core = (b_many + n_many * OPEN_COST) / slots
max_split      = min(MAX_PART, max(OPEN_COST, bytes_per_core))
per_partition  = int(max_split // OPEN_COST)
predicted      = math.ceil(n_many / per_partition)

print(f"openCostInBytes     : {OPEN_COST/1024/1024:>7.0f} MiB   (each file costs this much of the budget)")
print(f"maxPartitionBytes   : {MAX_PART/1024/1024:>7.0f} MiB")
print(f"slots               : {slots:>7}")
print(f"bytes per core      : {bytes_per_core/1024/1024:>7.0f} MiB")
print(f"-> maxSplitBytes    : {max_split/1024/1024:>7.0f} MiB")
print(f"-> files/partition  : {per_partition:>7}")
print(f"-> predicted        : {predicted:>7} partitions")
print(f"   measured         : {part_many:>7} partitions")
assert predicted == part_many, "the packing arithmetic no longer describes this reader"
print("\nThe prediction and the measurement agree, so the mechanism is the packing rule.")

openCostInBytes     :       4 MiB   (each file costs this much of the budget)
maxPartitionBytes   :     128 MiB
slots               :      18
bytes per core      :     268 MiB
-> maxSplitBytes    :     128 MiB
-> files/partition  :      32
-> predicted        :      38 partitions
   measured         :      38 partitions

The prediction and the measurement agree, so the mechanism is the packing rule.


### Where the folklore comes from

"One task per file" is not wrong about Spark; it is wrong about *this* reader. The older RDD
reader, `sc.textFile`, goes through Hadoop's input format, which never merges two files into one
split. The two are compared below on the same directory of tiny files.

The consequence for tuning is worth stating in the chapter's own terms: a small-files problem
reaches a DataFrame job through the **opening cost and the listing**, not through the task count.
The task count is capped by the packing rule, which is why the symptom below is tasks that are
*few and slow* rather than *thousands and instant*.

In [6]:
TINY = os.path.join(SCRATCH, "ch05-tiny-text")
shutil.rmtree(TINY, ignore_errors=True)
os.makedirs(TINY)
for i in range(60):
    with open(os.path.join(TINY, f"part-{i:04d}.txt"), "w") as f:
        f.write("alpha\nbeta\ngamma\n")

print("60 tiny text files")
print("  sc.textFile   (RDD reader, Hadoop splits) ->",
      sc.textFile(TINY).getNumPartitions(), "partitions")
print("  spark.read    (DataFrame reader, packed)  ->",
      spark.read.text(TINY).rdd.getNumPartitions(), "partitions")

print()
print("And the budget is a control, not a constant:")
for cost in ("4m", "1m", "64k"):
    spark.conf.set("spark.sql.files.openCostInBytes", cost)
    print(f"  openCostInBytes={cost:>4} -> {spark.read.parquet(MANY).rdd.getNumPartitions():>5}"
          f" read partitions over the {n_many:,} fragments")
spark.conf.set("spark.sql.files.openCostInBytes", f"{OPEN_COST}b")

60 tiny text files


  sc.textFile   (RDD reader, Hadoop splits) -> 60 partitions
  spark.read    (DataFrame reader, packed)  -> 15 partitions

And the budget is a control, not a constant:
  openCostInBytes=  4m ->    38 read partitions over the 1,200 fragments
  openCostInBytes=  1m ->    18 read partitions over the 1,200 fragments


  openCostInBytes= 64k ->    18 read partitions over the 1,200 fragments


## 3. The chapter's job, run

Three lines: read, filter, write partitioned by a column. The chapter's version filters
`ss_quantity > 1` and partitions the output by that column; this one filters `NumMentions > 1`
and partitions by `SQLDATE`, whose distinct-value count was printed above.

One line is added that the chapter's version does not show, and it is the one that makes the
write-side claim reproducible: `.repartition(200)` puts the data into two hundred in-memory
partitions, which is the number §5.8.1 argues from and the default shuffle count any real
pipeline would arrive at on its own.

In [7]:
OUT_BAD  = os.path.join(SCRATCH, "ch05-out-bad")
OUT_GOOD = os.path.join(SCRATCH, "ch05-out-good")
for p in (OUT_BAD, OUT_GOOD):
    shutil.rmtree(p, ignore_errors=True)

jobs_before = len(ui("/jobs"))
sc.setJobDescription("BAD: 200 in-memory partitions, written partitionBy(SQLDATE)")

t0 = time.time()
df = spark.read.parquet(MANY)
filtered = df.filter(F.col("NumMentions") > 1)
(filtered
 .repartition(200)
 .write.mode("overwrite")
 .partitionBy("SQLDATE")
 .parquet(OUT_BAD))
bad_secs = time.time() - t0

bad_files, bad_bytes = folder_stats(OUT_BAD)
bad_dirs = len([d for d in os.listdir(OUT_BAD) if d.startswith("SQLDATE=")])
bad_jobs = len(ui("/jobs")) - jobs_before

print(f"wall clock          : {bad_secs:6.1f} s")
print(f"output directories  : {bad_dirs:6}  (one per distinct SQLDATE)")
print(f"output files        : {bad_files:6}  ({bad_bytes/1e6:.1f} MB, {bad_bytes/bad_files/1e3:.1f} KB each)")
print(f"jobs launched       : {bad_jobs:6}")

wall clock          :    8.9 s
output directories  :     35  (one per distinct SQLDATE)
output files        :   5378  (25.6 MB, 4.8 KB each)
jobs launched       :      3


## 4. The top-down traversal

The chapter's sequence is fixed: job, then slowest stage, then the task distribution inside it,
then the executor that hosted the outlier. Each step below is one tab.

### Step 1 — the Jobs tab

In [8]:
def jobs_table(last=None):
    rows = []
    for j in ui("/jobs"):
        secs = None
        if j.get("completionTime") and j.get("submissionTime"):
            secs = (pd.Timestamp(j["completionTime"]) -
                    pd.Timestamp(j["submissionTime"])).total_seconds()
        rows.append({"job": j["jobId"],
                     "description": (j.get("description") or j["name"]).split("\n")[0][:44],
                     "stages": len(j["stageIds"]),
                     "tasks": j["numTasks"],
                     "seconds": None if secs is None else round(secs, 2)})
    rows.sort(key=lambda r: r["job"])
    return pd.DataFrame(rows if last is None else rows[-last:])

print(jobs_table(last=6).to_string(index=False))

 job                                  description  stages  tasks  seconds
  12              build the same data as one file       1      1     0.01
  13              build the same data as one file       1      1     0.01
  14              build the same data as one file       1      1     0.01
  15 BAD: 200 in-memory partitions, written parti       1      1     0.01
  16 BAD: 200 in-memory partitions, written parti       1     38     0.86
  17 BAD: 200 in-memory partitions, written parti       2    238     5.99


### Step 2 — the Stages tab, sorted by duration

The chapter says to sort by duration, which identifies the slowest stage immediately. Beside the
duration, the two columns that matter here are the task count and the shuffle volume.

In [9]:
def stage_labels():
    """stage id -> the job description that submitted it, which the stage name does not carry."""
    out = {}
    for j in ui("/jobs"):
        label = (j.get("description") or j["name"]).split("\n")[0][:40]
        for sid in j["stageIds"]:
            out[sid] = label
    return out

def stages_table(last=None):
    labels = stage_labels()
    rows = []
    for s in ui("/stages"):
        secs = None
        if s.get("completionTime") and s.get("firstTaskLaunchedTime"):
            secs = (pd.Timestamp(s["completionTime"]) -
                    pd.Timestamp(s["firstTaskLaunchedTime"])).total_seconds()
        rows.append({"stage": s["stageId"],
                     "submitted by": labels.get(s["stageId"], "-"),
                     "tasks": s["numTasks"],
                     "seconds": None if secs is None else round(secs, 2),
                     "in MB": round(s["inputBytes"] / 1e6, 1),
                     "shuf w MB": round(s["shuffleWriteBytes"] / 1e6, 1),
                     "shuf r MB": round(s["shuffleReadBytes"] / 1e6, 1),
                     "spill MB": round(s["diskBytesSpilled"] / 1e6, 1),
                     "GC ms": s["jvmGcTime"]})
    rows.sort(key=lambda r: r["stage"])
    df_ = pd.DataFrame(rows if last is None else rows[-last:])
    return df_.sort_values("seconds", ascending=False, na_position="last")

bad_stages = stages_table(last=3)
print(bad_stages.to_string(index=False))

 stage                             submitted by  tasks  seconds  in MB  shuf w MB  shuf r MB  spill MB  GC ms
    24 BAD: 200 in-memory partitions, written p    200     5.98    0.0        0.0       26.8       0.0   4919
    22 BAD: 200 in-memory partitions, written p     38     0.86   17.8       26.8        0.0       0.0   1291
    23 BAD: 200 in-memory partitions, written p     38      NaN    0.0        0.0        0.0       0.0      0


### Step 3 — the summary metrics of the slowest stage

This is the table the whole chapter turns on. Each row is one metric and each column a percentile
over that stage's tasks, so a row is read **across** rather than down.

The reading to take is the **median-to-maximum comparison** on the duration row. A maximum many
times the median is skew. A median and a maximum that are both negligible are an over-partitioned
stage. A nonzero spill row is memory pressure.

**This stage is none of those**, and that is worth seeing once. The comparison is a test, and a
test that always finds something is not a test. Here the tasks are balanced, substantial, and
spilling nothing — the stage is healthy by every reading the table offers, and the job is still
laying down thousands of fragments. The damage of a bad write is not visible in the metrics of
the job that does it; it is visible in the file count, and it is paid by every job that reads
the output afterwards. Notebook 5.3 builds a stage that *does* fail each of these readings.

In [10]:
SUMMARY_ROWS = [
    ("duration",                "Duration (ms)"),
    ("schedulerDelay",          "Scheduler delay (ms)"),
    ("executorDeserializeTime", "Task deserialization (ms)"),
    ("jvmGcTime",               "GC time (ms)"),
    ("resultSerializationTime", "Result serialization (ms)"),
    ("gettingResultTime",       "Getting result (ms)"),
    ("peakExecutionMemory",     "Peak execution memory (B)"),
    ("memoryBytesSpilled",      "Shuffle spill, memory (B)"),
    ("diskBytesSpilled",        "Shuffle spill, disk (B)"),
]

def summary_metrics(stage_id, attempt=0):
    q = "0,0.25,0.5,0.75,1.0"
    d = ui(f"/stages/{stage_id}/{attempt}/taskSummary?quantiles={q}")
    rows = [{"metric": label, "min": v[0], "25th": v[1],
             "median": v[2], "75th": v[3], "max": v[4]}
            for key, label in SUMMARY_ROWS if (v := d.get(key))]
    for group, label in (("shuffleReadMetrics", "Shuffle read (B)"),
                         ("shuffleWriteMetrics", "Shuffle write (B)")):
        g = d.get(group) or {}
        v = g.get("readBytes") or g.get("writeBytes")
        if v:
            rows.append({"metric": label, "min": v[0], "25th": v[1],
                         "median": v[2], "75th": v[3], "max": v[4]})
    out = pd.DataFrame(rows)
    for c in ("min", "25th", "median", "75th", "max"):
        out[c] = out[c].map(lambda x: f"{x:,.0f}")
    return out

slowest_bad = int(bad_stages.iloc[0]["stage"])
print(f"stage {slowest_bad}, the slowest of the run\n")
print(summary_metrics(slowest_bad).to_string(index=False))

sm = ui(f"/stages/{slowest_bad}/0/taskSummary?quantiles=0.5,1.0")["duration"]
ratio = sm[1] / max(sm[0], 1)
verdict = ("skew: the maximum is several times the median" if ratio >= 3 else
           "over-partitioned: both figures are negligible" if sm[1] < 50 else
           "neither skew nor over-partitioning -- the tasks are balanced and substantial")
print(f"\nmedian {sm[0]:,.0f} ms   maximum {sm[1]:,.0f} ms   ratio {ratio:.1f}x")
print("verdict:", verdict)

stage 24, the slowest of the run

                   metric       min      25th    median      75th       max
            Duration (ms)       320       496       519       542       694
     Scheduler delay (ms)         1         2         2         3         6
Task deserialization (ms)         8        10        11        16        45
             GC time (ms)         7        22        24        28        41
Result serialization (ms)         0         0         0         0         1
      Getting result (ms)         0         0         0         0         0
Peak execution memory (B) 1,179,632 1,179,632 1,179,632 1,179,632 1,179,632
Shuffle spill, memory (B)         0         0         0         0         0
  Shuffle spill, disk (B)         0         0         0         0         0
         Shuffle read (B)   132,525   133,511   133,765   134,132   134,854
        Shuffle write (B)         0         0         0         0         0

median 519 ms   maximum 694 ms   ratio 1.3x
verdict: 

### Step 4 — the Executors tab

The chapter's use for this page is comparative: one executor holding most of the shuffle volume
means the data is skewed toward keys it owns, and one holding most of the failures means the
machine is suspect. In `local[*]` there is a single executor, the driver, so the page has one
row — but the columns are the ones to learn, and the storage-memory column returns in notebook
5.4.

In [11]:
ex = pd.DataFrame([{"executor": e["id"],
                    "cores": e["totalCores"],
                    "max memory MB": round(e["maxMemory"] / 1e6, 1),
                    "memory used MB": round(e["memoryUsed"] / 1e6, 2),
                    "tasks done": e["completedTasks"],
                    "failed": e["failedTasks"],
                    "task time s": round(e["totalDuration"] / 1000, 1),
                    "GC time s": round(e["totalGCTime"] / 1000, 1),
                    "shuffle read MB": round(e["totalShuffleRead"] / 1e6, 1),
                    "shuffle write MB": round(e["totalShuffleWrite"] / 1e6, 1)}
                   for e in ui("/executors")])
print(ex.to_string(index=False))

row = ex.iloc[0]
share = row["GC time s"] / max(row["task time s"], 0.001)
print(f"\nGC as a share of task time: {share:6.1%}"
      f"   ({'above' if share > 0.1 else 'below'} the chapter's one-tenth threshold)")

executor  cores  max memory MB  memory used MB  tasks done  failed  task time s  GC time s  shuffle read MB  shuffle write MB
  driver     18          455.5             2.2        1505       0         19.6        0.6             72.8              72.8

GC as a share of task time:   3.1%   (below the chapter's one-tenth threshold)


### The SQL tab

For DataFrame work the stage view is one level too low: it says what ran without saying which
part of the query it belongs to. The SQL tab supplies the correspondence, and the node to count
is `Exchange`, because each one is a shuffle and therefore a stage boundary.

In [12]:
queries = ui("/sql")
q = sorted(queries, key=lambda x: x["id"])[-1]
print("description :", q["description"].split("\n")[0][:70])
print("duration    :", q["duration"], "ms")
print("jobs        :", list(q.get("successJobIds", [])))

plan = (filtered.repartition(200)._jdf.queryExecution().executedPlan().toString())
print("\nphysical plan of the same computation:\n")
for line in plan.split("\n")[:12]:
    print("   ", line[:110])
print("\nExchange nodes (each one is a shuffle):", plan.count("Exchange"))

description : BAD: 200 in-memory partitions, written partitionBy(SQLDATE)
duration    : 8803 ms
jobs        : [16, 17]

physical plan of the same computation:

    AdaptiveSparkPlan isFinalPlan=false
    +- Exchange RoundRobinPartitioning(200), REPARTITION_BY_NUM, [plan_id=340]
       +- Filter (isnotnull(NumMentions#125) AND (NumMentions#125 > 1))
          +- FileScan parquet [GLOBALEVENTID#121L,SQLDATE#122,EventRootCode#123,QuadClass#124,NumMentions#125,AvgT
    

Exchange nodes (each one is a shuffle): 1


## 5. The remedy

The chapter is explicit that the remedy follows from the diagnosis rather than from a setting.
The number of files written is the number of partitions in memory at the moment of the write,
and with `partitionBy` each of those partitions additionally writes one file for **every distinct
value it happens to contain**. Two hundred partitions, each holding a scattering of every
date, therefore write up to 200 × (number of dates) fragments.

Repartitioning by the same column collects all records of one date into one task, so that task
writes one file rather than a fragment of each.

In [13]:
sc.setJobDescription("GOOD: repartition(SQLDATE) before the same write")
t0 = time.time()
(filtered
 .repartition("SQLDATE")
 .write.mode("overwrite")
 .partitionBy("SQLDATE")
 .parquet(OUT_GOOD))
good_secs = time.time() - t0

good_files, good_bytes = folder_stats(OUT_GOOD)
good_dirs = len([d for d in os.listdir(OUT_GOOD) if d.startswith("SQLDATE=")])

print(f"wall clock          : {good_secs:6.1f} s")
print(f"output directories  : {good_dirs:6}")
print(f"output files        : {good_files:6}  ({good_bytes/1e6:.1f} MB,"
      f" {good_bytes/good_files/1e3:.1f} KB each)")

good_stages = stages_table(last=2)
slowest_good = int(good_stages.iloc[0]["stage"])
print(f"\nsummary metrics of stage {slowest_good}, the write stage:\n")
print(summary_metrics(slowest_good).to_string(index=False))

wall clock          :    0.5 s
output directories  :     35
output files        :     35  (7.7 MB, 221.1 KB each)

summary metrics of stage 27, the write stage:

                   metric       min       25th     median       75th        max
            Duration (ms)        99        102        144        153        163
     Scheduler delay (ms)         1          1          2          2          2
Task deserialization (ms)        15         15         15         16         17
             GC time (ms)         7          7          8          8          8
Result serialization (ms)         0          0          0          0          0
      Getting result (ms)         0          0          0          0          0
Peak execution memory (B) 8,388,512 11,534,224 13,631,344 15,728,464 16,777,024
Shuffle spill, memory (B)         0          0          0          0          0
  Shuffle spill, disk (B)         0          0          0          0          0
         Shuffle read (B) 2,204,282  2

## 6. The two runs side by side

This is the comparison the proposed notebook exists to make. The timings are machine-dependent
and in `local[*]` they understate the difference badly, because every "network" fetch is a local
disk read and object storage is not involved at all. The **file counts** are the figures that
transfer, because they are what every later reader of this output will pay for.

In [14]:
def stage_of(stage_id):
    s = [x for x in ui("/stages") if x["stageId"] == stage_id][0]
    d = ui(f"/stages/{stage_id}/0/taskSummary?quantiles=0.5,1.0")
    return s, d

sb, db = stage_of(slowest_bad)
sg, dg = stage_of(slowest_good)

comparison = pd.DataFrame([
    {"": "wall clock (s)",           "bad": round(bad_secs, 1),  "good": round(good_secs, 1)},
    {"": "output files",             "bad": bad_files,           "good": good_files},
    {"": "files per date directory", "bad": round(bad_files / bad_dirs, 1),
                                     "good": round(good_files / good_dirs, 1)},
    {"": "mean output file KB",      "bad": round(bad_bytes / bad_files / 1e3, 1),
                                     "good": round(good_bytes / good_files / 1e3, 1)},
    {"": "tasks in the write stage", "bad": sb["numTasks"],      "good": sg["numTasks"]},
    {"": "median task (ms)",         "bad": round(db["duration"][0]),
                                     "good": round(dg["duration"][0])},
    {"": "max task (ms)",            "bad": round(db["duration"][1]),
                                     "good": round(dg["duration"][1])},
    {"": "shuffle read MB",          "bad": round(sb["shuffleReadBytes"] / 1e6, 1),
                                     "good": round(sg["shuffleReadBytes"] / 1e6, 1)},
])

# "lower is better" for every row except the mean file size, where bigger files are the win.
lower_is_better = [True, True, True, False, True, True, True, True]
factors = []
for b, g, low in zip(comparison["bad"], comparison["good"], lower_is_better):
    if not g or not b:
        factors.append("-")
    else:
        factors.append(f"{b / g:.1f}x worse" if low else f"{g / b:.1f}x worse")
comparison["the bad run is"] = factors
print(comparison.to_string(index=False))

print(f"\nOne line of code changed the output from {bad_files:,} files to {good_files},")
print(f"which is exactly one per date directory.")
print(f"Projected to a year of daily data at 200 in-memory partitions:"
      f" {200 * 365:,} files, which is the chapter's 'on the order of seventy thousand'.")

                            bad  good the bad run is
          wall clock (s)    8.9   0.5    17.8x worse
            output files 5378.0  35.0   153.7x worse
files per date directory  153.7   1.0   153.7x worse
     mean output file KB    4.8 221.1    46.1x worse
tasks in the write stage  200.0   7.0    28.6x worse
        median task (ms)  519.0 144.0     3.6x worse
           max task (ms)  694.0 163.0     4.3x worse
         shuffle read MB   26.8  23.2     1.2x worse

One line of code changed the output from 5,378 files to 35,
which is exactly one per date directory.
Projected to a year of daily data at 200 in-memory partitions: 73,000 files, which is the chapter's 'on the order of seventy thousand'.


## 7. The cheapest check of all: count the jobs

Before any of the above, §5.2.2 gives a check that costs nothing: the number of jobs should equal
the number of actions. A program showing more jobs than it has actions has triggered an implicit
one, and schema inference on a CSV read is the commonest cause — Spark must read part of the file
before it can plan a query over it.

Two reads of the same file below, one with a declared schema and one without. Neither is an
action. One of them launches jobs anyway.

In [15]:
CSV = os.path.join(DATA, "taxi-data-sorted-verysmall-header.csv")

before = len(ui("/jobs"))
declared = (spark.read.option("header", "true")
            .schema("medallion string, hack_license string, pickup_datetime string,"
                    " dropoff_datetime string, trip_time int, trip_distance double,"
                    " pickup_longitude double, pickup_latitude double,"
                    " dropoff_longitude double, dropoff_latitude double,"
                    " payment_type string, fare_amount double, surcharge double,"
                    " mta_tax double, tip_amount double, tolls_amount double,"
                    " total_amount double")
            .csv(CSV))
after_declared = len(ui("/jobs"))

inferred = (spark.read.option("header", "true").option("inferSchema", "true").csv(CSV))
after_inferred = len(ui("/jobs"))

print(f"declared schema, no action : {after_declared - before} job(s)")
print(f"inferSchema,     no action : {after_inferred - after_declared} job(s)")
print()
print("Same file, same number of actions written by the programmer: zero.")
print("The second read is a job in the Jobs tab that corresponds to no line the programmer")
print("would call an action, which is exactly the discrepancy the chapter says to look for.")

declared schema, no action : 0 job(s)
inferSchema,     no action : 2 job(s)

Same file, same number of actions written by the programmer: zero.
The second read is a job in the Jobs tab that corresponds to no line the programmer
would call an action, which is exactly the discrepancy the chapter says to look for.


## Conclusion

The traversal the chapter prescribes was followed end to end and it worked: the Jobs tab named
the expensive job, the Stages tab named the stage, the summary metrics gave the task
distribution inside it, and the Executors tab said whether one machine was carrying the load.
Every one of those readings came out of the same JSON the browser renders, which is what makes
them reproducible here.

Three things the run established that the prose could not:

1. **The file count is the durable cost, not the clock.** One line moved the output from
   thousands of fragments to one file per date. In `local[*]` the wall clock barely notices; on
   object storage, where every file is a request with its own latency, every later reader pays
   for the difference.
2. **A small-files problem does not reach a DataFrame job as one task per file.** The reader
   packs the fragments, charging each one `openCostInBytes` against a `maxSplitBytes` budget, so
   1,200 files became a few dozen partitions and the arithmetic predicted the count exactly. The
   one-task-per-file rule belongs to `sc.textFile` and the Hadoop input formats, which is where
   the folklore comes from. The cost of small files is real and is paid in listing, opening and
   write amplification.
3. **The job count is the cheapest diagnostic in the chapter.** A read with `inferSchema`
   launches a job while the programmer is still writing the query, and nothing in the source
   says so.

*Chapter section:* §5.2 (the Spark web UI), §5.2.7 (the worked diagnosis), §5.8.1 (partitioning
on write). *Exercise 1* is answered from the summary-metrics table of section 3 above; notebook
5.3 builds both of its stages deliberately.